# 03 — Comparing diabetes groups in PAD000040

**Dataset:** *Inflammatory proteomic heterogeneity beyond glycemia status in severe obesity.*
Milito MM, ..., Giraudi PJ. Int J Mol Sci 2026;27(9):4152. doi:10.3390/ijms27094152.
PRIDE: [PAD000040](https://www.ebi.ac.uk/pride/archive/projects/PAD000040), licence CC0.

**Question.** Do the 92 inflammation proteins differ between people with no diabetes, prediabetes and
type 2 diabetes (T2DM)?

**Route.** Notebooks 01 and 02 prepared the data; this notebook runs the whole route in one place
and ends with the statistics:

1. Read both plates, attach the sample list, quality control, drop the broken sample 249.
2. Bridge the plates (prediabetes was measured only on plate 2 and on the bridge samples).
3. `to_wide`: one row per person (the bridge samples' two measurements are averaged).
4. `compare_groups`: a trend over the three groups, then the three groups without an order, then
   T2DM against no diabetes.
5. Adjusted and unadjusted volcano plots side by side.

In [1]:
from pathlib import Path
import pandas as pd

from signatomics.olink import read_npx, add_sample_info, apply_qc, bridge_plates, to_wide
from signatomics.stats import compare_groups, plot_volcano, plot_volcano_pair

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 160)

here = Path.cwd()
ROOT = next(p for p in [here, *here.parents, here / "signatomics"] if (p / "src" / "signatomics").is_dir())
DATA = ROOT / "data" / "PAD000040"

## 1. Prepare the data

The two plate files are read separately and put in one table. `add_sample_info` attaches the sample
list (diabetes status, sex). Sample 249 on plate 2 is the broken bridge measurement found in notebook
02; `apply_qc(drop_samples=...)` removes it and reports the removal.

In [2]:
plates = pd.concat([read_npx(DATA / "inflam_NASH_dt1.raw.csv", verbose=False),
                    read_npx(DATA / "inflam_PRA_dt2.raw.csv", verbose=False)], ignore_index=True)
data = add_sample_info(plates, DATA / "2025.05.08_sample_manifest.xlsx", on="FIF ID", verbose=False)
clean = apply_qc(data, drop_samples={"249": "Inflammation PRAESIIDIUM Giraudi 04_03_2025"}, show_ids=False)

Quality steps applied
  removed 1 measurement (chosen by you (drop_samples)): 1 (IDs hidden)
  removed 2 measurements (no NPX value at all): 2 (IDs hidden)
  kept: 175 measurements of 143 samples, 92 assays


## 2. Bridge the plates, then make the analysis table

Bridging comes first. Without it, a plate difference of about +0.5 NPX on plate 2 would look like a
prediabetes effect, because prediabetes was measured on plate 2 and on the bridge samples only.

In [3]:
bridged = bridge_plates(clean)
table = to_wide(bridged)
table[["diabetes", "sex"]].value_counts().unstack()

Bridging report
  reference (kept as it is): Inflammation F.I.F 21/03/2024
  Inflammation PRAESIIDIUM Giraudi 04_03_2025: 92 assays adjusted by their adjustment factor (median -0.55 NPX, from -2.31 to +0.93), each from 28 bridge samples
  NPX and LOD shifted together, so 'below the LOD' stays the same; the original values are in NPX_unbridged and LOD_unbridged
  check: plot_bridge_pca(before, after) - each bridge sample's two measurements should now lie together
Analysis table
  averaged: 32 samples measured more than once (bridge samples or repeats) get the average of their NPX values
  result: 143 samples x 92 assays, plus 10 columns of sample metadata (FIF ID, PlateID_samplelist, Subject_ID, sex, diabetes, fibrosi, fibrosis_diag, inflammation_diag, ballooning_diag, steatosis_diagnosis)


sex,F,M
diabetes,,
T2DM,26,22
Prediabetes,17,8
No diabetes,62,7


The table has one row per person. Sex is unevenly spread over the groups: 62 of 69 people without
diabetes are women, against 26 of 48 with T2DM. Section 5 comes back to this.

In [4]:
order = ["No diabetes", "Prediabetes", "T2DM"]

## 3. Three groups in a natural order: the trend

The groups run from no diabetes through prediabetes to T2DM, so one trend test asks the sharpest
question: does a protein rise (or fall) step by step? `ordered=True` fits a straight line through the
three group means with robust (HC3) standard errors. The `slope` is the change in NPX per step.

In [5]:
trend = compare_groups(table, "diabetes", reference="No diabetes", ordered=True, order=order)
trend.table.head(6)[["protein", "slope", "ci_low", "ci_high", "p", "fdr"]]

Group comparison
  test: linear trend over the group steps (robust HC3 standard errors); order: No diabetes < Prediabetes < T2DM
  groups: No diabetes 69, Prediabetes 25, T2DM 48 (reference: No diabetes)
  left out: 1 sample without a value in 'diabetes'
  tested: 92 proteins
  FDR below 0.05: 3 proteins: IL8 (+0.30 NPX, FDR 0.016), Flt3L (+0.23 NPX, FDR 0.016), CDCP1 (+0.23 NPX, FDR 0.033)
  pairwise: read .pairwise (p_holm) for the proteins with FDR below 0.05


,protein,slope,ci_low,ci_high,p,fdr
0,IL8,0.302872,0.141510,0.464233,0.000297,0.015747
1,Flt3L,0.233102,0.107575,0.358629,0.000342,0.015747
2,CDCP1,0.233078,0.095218,0.370938,0.001065,0.032654
3,CCL20,0.232121,0.071012,0.393229,0.005056,0.097411
4,HGF,0.164200,0.049607,0.278793,0.005294,0.097411
5,IL-15RA,0.091197,0.024957,0.157436,0.007315,0.112161


**How to read this.** Three proteins have an FDR below 0.05: IL8, Flt3L and CDCP1. Each rises by
roughly 0.2 to 0.3 NPX per step, so T2DM sits about 0.5 NPX (1.4 times) above no diabetes. The FDR
corrects for testing 92 proteins at once.

In [6]:
plot_volcano(trend)

Each dot is one protein. The x axis shows the slope, the y axis how certain it is. The dotted line is
the p-value of the weakest protein that passed the FDR.

## 4. Three groups without an order, and the pairs

Without `ordered=True` the test is Welch's ANOVA: does any group differ? It also compares every pair
of groups, Holm-adjusted within each protein, in `.pairwise`.

In [7]:
anova = compare_groups(table, "diabetes", reference="No diabetes")

Group comparison
  test: Welch's ANOVA; then every pair of groups, Holm-adjusted within each protein
  groups: No diabetes 69, Prediabetes 25, T2DM 48 (reference: No diabetes)
  left out: 1 sample without a value in 'diabetes'
  tested: 92 proteins
  FDR below 0.05: none (lowest FDR 0.09)
  pairwise: read .pairwise (p_holm) for the proteins with FDR below 0.05


In [8]:
anova.pairwise.query("protein in ['IL8', 'Flt3L', 'CDCP1']").round(3)

,protein,group,reference,difference,ci_low,ci_high,p,p_holm,omnibus_significant
0,IL8,Prediabetes,No diabetes,0.463,0.065,0.862,0.024,0.047,False
1,IL8,T2DM,No diabetes,0.595,0.269,0.921,0.000,0.001,False
2,IL8,T2DM,Prediabetes,0.132,-0.318,0.581,0.560,0.560,False
15,CDCP1,Prediabetes,No diabetes,0.126,-0.175,0.428,0.402,0.402,False
16,CDCP1,T2DM,No diabetes,0.473,0.194,0.753,0.001,0.004,False
17,CDCP1,T2DM,Prediabetes,0.347,-0.013,0.707,0.059,0.117,False
186,Flt3L,Prediabetes,No diabetes,0.286,0.003,0.569,0.047,0.095,False
187,Flt3L,T2DM,No diabetes,0.463,0.213,0.713,0.000,0.001,False
188,Flt3L,T2DM,Prediabetes,0.176,-0.117,0.469,0.233,0.233,False


**How to read this.** The any-group test finds none at FDR 0.05 (lowest FDR 0.09). It spends its
power on every possible pattern; the trend test spends it on one. Both are valid questions, and the
choice is made before looking at the results. For IL8, Flt3L and CDCP1 the pairs show that T2DM
differs from no diabetes (Holm p of 0.001 to 0.004). Prediabetes sits in between, closer to no diabetes for
CDCP1 and closer to T2DM for IL8.

Pairs are meant to be read for proteins that pass the overall test, so treat these as a description
of the trend result.

## 5. Adjusted and unadjusted

The two groups with the clearest contrast are no diabetes and T2DM. The sample list holds one
covariate, sex. `covariates=["sex"]` fits `NPX ~ diabetes + sex` for every protein, with the same model
for all of them and no covariate chosen by its p-value.

In [9]:
two = table[table["diabetes"].isin(["No diabetes", "T2DM"])]
unadjusted = compare_groups(two, "diabetes", reference="No diabetes")
adjusted = compare_groups(two, "diabetes", reference="No diabetes", covariates=["sex"])

Group comparison
  test: Welch's t-test; effect: difference in mean NPX (= log2 fold change), T2DM minus No diabetes, with 95% interval
  groups: No diabetes 69, T2DM 48 (reference: No diabetes)
  tested: 92 proteins
  FDR below 0.05: 3 proteins: Flt3L (+0.46 NPX, FDR 0.022), IL8 (+0.59 NPX, FDR 0.022), CDCP1 (+0.47 NPX, FDR 0.036)


Group comparison (adjusted)
  test: linear model, NPX ~ diabetes + sex; effect: difference in mean NPX (= log2 fold change), T2DM minus No diabetes
  groups: No diabetes 69, T2DM 48 (reference: No diabetes)
  adjusted for: sex (one model for every protein, robust HC3 standard errors; no covariate is chosen or dropped by its p-value)
  tested: 92 proteins
  FDR below 0.05: none (lowest FDR 0.81)


In [10]:
plot_volcano_pair(unadjusted, adjusted)

In [11]:
both = (unadjusted.table.set_index("protein")[["difference", "p", "fdr"]]
        .join(adjusted.table.set_index("protein")[["difference", "p", "fdr"]],
              lsuffix="_unadjusted", rsuffix="_adjusted"))
both.head(6).round(3)

,difference_unadjusted,p_unadjusted,fdr_unadjusted,difference_adjusted,p_adjusted,fdr_adjusted
protein,,,,,,
Flt3L,0.463,0.000,0.022,0.381,0.010,0.807
IL8,0.595,0.000,0.022,0.404,0.036,0.807
CDCP1,0.473,0.001,0.036,0.376,0.019,0.807
CCL20,0.461,0.006,0.116,0.309,0.085,0.807
HGF,0.326,0.006,0.116,0.205,0.069,0.807
IL-15RA,0.180,0.009,0.143,0.108,0.094,0.807


**How to read this.** Unadjusted, three proteins reach FDR 0.05 (Flt3L, IL8, CDCP1). After adjusting
for sex, the differences shrink by 18 to 32% (IL8 from +0.60 to +0.40) and the intervals widen, so no protein reaches FDR 0.05. Only 7 people without diabetes are men,
so the data cannot separate "T2DM" from "male" well. The direction stays the same for the three
proteins. The honest summary: a difference of about +0.4 to +0.6 NPX with T2DM, and this study is too small to
tell how much of it sex explains.

A study with age, BMI and medication in the sample list would give a fuller adjusted analysis. The
route is the same: name the covariates, compare the two volcano plots.

## Summary

- After bridging and removing sample 249, 142 people with a diabetes status remain (69 without diabetes, 25 with prediabetes, 48 with T2DM).
- The trend test finds IL8, Flt3L and CDCP1 rising with glycemia status (FDR below 0.05).
- Welch's ANOVA, which allows any pattern, finds no protein at FDR 0.05.
- T2DM against no diabetes: the same three proteins unadjusted; none after adjusting for sex, with
  the direction unchanged.

**Next:** the biomarker signature (`find_signature`), which uses these same proteins in a model and
tests how well they predict the group in people it has not seen.